In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import pickle

In [2]:
df = pd.read_csv("cleaned_data.csv")

In [3]:
X = df.drop(columns = "price")
y = df["price"]

In [35]:
X

,year,manufacturer,condition,fuel,odometer,title_status,transmission,drive,type,paint_color,car_age
0,2014.0,gmc,good,gas,57923.0,clean,other,4wd,pickup,white,7.0
1,2010.0,chevrolet,good,gas,71229.0,clean,other,4wd,pickup,blue,11.0
2,2020.0,chevrolet,good,gas,19160.0,clean,other,4wd,pickup,red,1.0
3,2017.0,toyota,good,gas,41124.0,clean,other,fwd,pickup,red,4.0
4,2013.0,ford,excellent,gas,128000.0,clean,automatic,rwd,truck,black,8.0
...,...,...,...,...,...,...,...,...,...,...,...
366111,2019.0,nissan,good,gas,32226.0,clean,other,fwd,sedan,unknown,2.0
366112,2020.0,volvo,good,gas,12029.0,clean,other,fwd,sedan,red,1.0
366113,2020.0,cadillac,good,diesel,4174.0,clean,other,4wd,hatchback,white,1.0
366114,2018.0,lexus,good,gas,30112.0,clean,other,fwd,sedan,silver,3.0


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .25, shuffle = True, random_state = 42)

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [6]:
X_train["condition"].unique()

array(['good', 'unknown', 'excellent', 'fair', 'like new', 'salvage',
       'new'], dtype=object)

In [7]:
le = le.fit(X_train["condition"])

In [8]:
X_train["condition"] = le.transform(X_train["condition"])
X_test["condition"] = le.transform(X_test["condition"])

In [9]:
objCols = [cols for cols in X_train.columns if X_train[cols].dtype == "object"]
numCols = [cols for cols in X_train.columns if X_train[cols].dtype != "object"]

In [10]:
numCols.remove("condition")
numCols

['year', 'odometer', 'car_age']

In [11]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [12]:
CT = ColumnTransformer([
    ("OHEncoder", OneHotEncoder(), objCols),
    ("Scaler", StandardScaler(), numCols)
], remainder = "passthrough")

In [13]:
X_train = CT.fit_transform(X_train)
X_test = CT.transform(X_test)

In [16]:
with open("Label_Encoder.pkl", "wb")as file:
    pickle.dump(le, file)
with open("Scaler_and_OneHotEncoder.pkl", "wb") as file:
    pickle.dump(CT, file)

In [25]:
X_train

<274587x90 sparse matrix of type '<class 'numpy.float64'>'
	with 2953419 stored elements in Compressed Sparse Row format>

In [18]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
Models = {
    "Linear" : LinearRegression(), 
    "Ridge" : Ridge(), 
    "Lasso" : Lasso(), 
    "ElasticNet" : ElasticNet(),
    "Decision Tree" : DecisionTreeRegressor(),
    "Random Forest" : RandomForestRegressor(max_depth = 15, n_jobs = -1),
    "AdaBoost" : AdaBoostRegressor(),
    "Gradient Boost" : GradientBoostingRegressor(),
    "XG Boost" : XGBRegressor()
}

In [16]:
def modelEvaluation(y_pred, y_test):
    print("mse :", mean_squared_error(y_pred, y_test))
    print("mae :", mean_absolute_error(y_pred, y_test))
    print("r2 :", r2_score(y_test, y_pred))

In [17]:
for models in (list(Models)):
    print(models)
    model = Models[models]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    modelEvaluation(y_pred, y_test)

Linear
mse : 85638702.92320845
mae : 6742.29010639559
r2 : 0.46374843070550764
Ridge
mse : 85640949.22503681
mae : 6742.2407705465885
r2 : 0.46373436483529407
Lasso
mse : 85661523.04820155
mae : 6742.8348667522405
r2 : 0.4636055358761684
ElasticNet
mse : 106577606.63300899
mae : 8142.63167900028
r2 : 0.3326334138917274
Decision Tree
mse : 34870036.714631096
mae : 2548.2678047010195
r2 : 0.7816511545446357
Random Forest
mse : 36073434.75610985
mae : 3817.232222291207
r2 : 0.7741157287826698
AdaBoost
mse : 76415288.23245049
mae : 6789.043289403114
r2 : 0.5215035161206607
Gradient Boost
mse : 51055768.15369386
mae : 4853.960553494863
r2 : 0.6802995040863196
XG Boost
mse : 36840468.08907345
mae : 3964.8549899040686
r2 : 0.7693126797676086


In [18]:
rf_params = {
    "max_depth" : [5, 8, 10, 15, 20],
    "max_features" : [5, 7, "auto", 8],
    "min_samples_split" : [2, 8, 15, 20],
    "n_estimators" : [100, 200, 500, 1000]
}
dt_params = {
    "criterion" : ["squared_error", "absolute_error", "poisson"], 
    "splitter" : ["best", "random"], 
    "max_depth" : [1, 2, 3, 4, 5, 10, 15, 20, 25], 
    "max_features" : ["sqrt", "log2", None]
}
gradient_params = {
    "loss" : ["squared_error", "huber", "absolute_error"],
    "criterion" : ["friedman_mse", "squared_error", "mse"], 
    "min_samples_split" : [2, 8, 15, 20],
    "n_estimators" : [100, 200, 500, 1000],
    "max_depth" : [5, 8, 10, 15, None],
    "learning_rate" : [.1, .01, .02, .03]
}
xgboost_params = {
    "learning_rate" : [.1, .01],
    "max_depth" : [5, 8, 12, 20, 30],
    "n_estimators" : [100, 200, 300],
    "colsample_bytree" : [.5, .8, 1, .3, .4]
}

In [19]:
Params = [(DecisionTreeRegressor(), dt_params),
          (GradientBoostingRegressor(), gradient_params),
          (XGBRegressor(), xgboost_params),
          (RandomForestRegressor(), rf_params)]

In [21]:
from sklearn.model_selection import RandomizedSearchCV
for model, params in Params:
    random = RandomizedSearchCV(estimator = model, param_distributions = params, cv = 3, n_iter = 10, verbose = 3, n_jobs = -1)
    random.fit(X_train, y_train)
    print(random.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'splitter': 'random', 'max_features': 'log2', 'max_depth': 20, 'criterion': 'poisson'}
Fitting 3 folds for each of 10 candidates, totalling 30 fits


C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
9 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sk

{'n_estimators': 1000, 'min_samples_split': 8, 'max_depth': 10, 'loss': 'squared_error', 'learning_rate': 0.1, 'criterion': 'friedman_mse'}
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'n_estimators': 300, 'max_depth': 20, 'learning_rate': 0.1, 'colsample_bytree': 0.3}
Fitting 3 folds for each of 10 candidates, totalling 30 fits


C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
9 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
7 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\keshv\anaconda3\Lib\site-packages\sk

{'n_estimators': 200, 'min_samples_split': 8, 'max_features': 8, 'max_depth': 20}


In [22]:
random.best_params_

{'n_estimators': 200,
 'min_samples_split': 8,
 'max_features': 8,
 'max_depth': 20}

In [26]:
Models = {"Random Forest" : RandomForestRegressor(n_estimators = 200, min_samples_split = 8, max_features = 8, max_depth = 20),
         "Decision Tree" : DecisionTreeRegressor(splitter = "random", max_features = "log2", max_depth = 20, criterion = "poisson"),
         "Gradient Boost" : GradientBoostingRegressor(n_estimators = 1000, min_samples_split = 8, max_depth = 10, loss = "squared_error", learning_rate = .1, criterion = "friedman_mse"),
         "XG Boost" : XGBRegressor(n_estimators = 300, max_depth = 20, learning_rate = .1, colsample_bytree = .3)}

In [27]:
for models in (list(Models)):
    print(models)
    model = Models[models]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    modelEvaluation(y_pred, y_test)

Random Forest
mse : 34481054.51177895
mae : 3865.9181943887857
r2 : 0.7840868793931792
Decision Tree
mse : 91652140.56338714
mae : 7068.863366580357
r2 : 0.42609354732536153
Gradient Boost
mse : 22378029.867258925
mae : 2781.9815182473003
r2 : 0.859873477476683
XG Boost
mse : 20537163.746305853
mae : 2346.147191655701
r2 : 0.8714005947113037


In [22]:
model = XGBRegressor(n_estimators = 300, max_depth = 20, learning_rate = .1, colsample_bytree = .3)
model = model.fit(X_train, y_train)

In [24]:
import joblib
joblib.dump(model, "car_price_model.pkl")

['car_price_model.pkl']